In [1]:
import numpy as np
import pandas as pd


In [ ]:
# Daten einlesen

df = pd.read_csv("ai4i2020.csv")

df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
# Features: Torque und Rotational Speed — standardisieren und Interaktion erstellen
# Wir verwenden die Originalspalten aus ai4i2020.csv: 'Torque [Nm]' und 'Rotational speed [rpm]'
try:
    df
except NameError:
    df = pd.read_csv("ai4i2020.csv")

cols = ['Torque [Nm]', 'Rotational speed [rpm]']
print('Prüfe Verfügbarkeit der Spalten:', [c for c in cols if c in df.columns])

# Standardisieren (z-score)
df['Torque_z'] = (df['Torque [Nm]'] - df['Torque [Nm]'].mean()) / df['Torque [Nm]'].std()
df['RotSpeed_z'] = (df['Rotational speed [rpm]'] - df['Rotational speed [rpm]'].mean()) / df['Rotational speed [rpm]'].std()

# Interaktionsterm
df['Torque_x_RotSpeed'] = df['Torque_z'] * df['RotSpeed_z']

df[['Torque [Nm]','Rotational speed [rpm]','Torque_z','RotSpeed_z','Torque_x_RotSpeed']].head()


Prüfe Verfügbarkeit der Spalten: ['Torque [Nm]', 'Rotational speed [rpm]']


,Torque [Nm],Rotational speed [rpm],Torque_z,RotSpeed_z,Torque_x_RotSpeed
0,42.8,1551,0.282186,0.068182,0.019240
1,46.3,1408,0.633276,-0.729435,-0.461934
2,49.4,1498,0.944242,-0.227438,-0.214757
3,39.5,1433,-0.048843,-0.589992,0.028817
4,40.0,1408,0.001313,-0.729435,-0.000958


In [9]:
# Designmatrix in Matrix-Schreibweise

# Standardisiere Features
df['Torque_z'] = (df['Torque [Nm]'] - df['Torque [Nm]'].mean()) / df['Torque [Nm]'].std()
df['RotSpeed_z'] = (df['Rotational speed [rpm]'] - df['Rotational speed [rpm]'].mean()) / df['Rotational speed [rpm]'].std()

# Anzahl Beobachtungen
n = df.shape[0]

# Designmatrix X: Spalten = [1 (Intercept), Torque_z, RotSpeed_z]
X = np.column_stack([
    np.ones(n),
    df['Torque_z'].to_numpy(),
    df['RotSpeed_z'].to_numpy()
])

# Zielvektor y (PWF)
y = df['PWF'].to_numpy()

# Kurze Ausgabe zur Kontrolle
print('X shape:', X.shape)
print('y shape:', y.shape)
X[:5]


X shape: (10000, 3)
y shape: (10000,)


array([[ 1.        ,  0.28218565,  0.06818173],
       [ 1.        ,  0.63327635, -0.72943503],
       [ 1.        ,  0.94424241, -0.22743847],
       [ 1.        , -0.04884274, -0.58999154],
       [ 1.        ,  0.00131308, -0.72943503]])

In [10]:
# Sigmoid- und Logloss-Funktion (Matrix-Formalismus, deutsch kommentiert)

# Sigmoid-Funktion: wählt Wahrscheinlichkeiten aus linearen Scores
def sigmoid(z):
    # numerisch stabil: clippen der Werte bevor exp gerechnet wird
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

# Logloss (negative log-likelihood) in Matrix-Form:
# L(theta) = - (1/n) * [ y^T log(p) + (1-y)^T log(1-p) ] mit p = sigmoid(X @ theta)
def logloss(theta, X, y, eps=1e-15):
    n = X.shape[0]
    z = X.dot(theta)            # linearer Score (n,)
    p = sigmoid(z)              # Wahrscheinlichkeiten (n,)
    # numerische Stabilität: eps vermeiden von log(0)
    p = np.clip(p, eps, 1 - eps)
    loss = - (1.0 / n) * (y.dot(np.log(p)) + (1 - y).dot(np.log(1 - p)))
    return loss

# Test mit Null-Parameter (Intercept=0, Koeffizienten=0)
theta0 = np.zeros(X.shape[1])
initial_loss = logloss(theta0, X, y)
print('Initialer Logloss (theta=0):', initial_loss)


Initialer Logloss (theta=0): 0.693147180559947


In [12]:
# Gradient Descent für logistische Regression (Matrix-Formalismus, deutsch kommentiert)

# Gradient der Logloss-Funktion (Matrixform)
def gradient(theta, X, y):
    n = X.shape[0]
    p = sigmoid(X.dot(theta))
    # Gradientenformel: (1/n) * X^T (p - y)
    return (1.0 / n) * X.T.dot(p - y)

# Einfacher Gradient Descent Optimierer
def gradient_descent(X, y, lr=0.5, max_iter=10000, tol=1e-7, verbose=True):
    theta = np.zeros(X.shape[1])
    losses = []
    for i in range(max_iter):
        g = gradient(theta, X, y)
        theta -= lr * g
        loss = logloss(theta, X, y)
        losses.append(loss)
        if verbose and (i % 1000 == 0):
            print(f'Iter {i}: loss={loss:.6f}')
        if np.linalg.norm(g) < tol:
            if verbose:
                print(f'Konvergenz bei Iteration {i}, ||grad||={np.linalg.norm(g):.3e}')
            break
    return theta, np.array(losses)

# Ausführen des GD mit Standardparametern
theta_gd, losses = gradient_descent(X, y, lr=0.5, max_iter=10000, tol=1e-7, verbose=True)
print('Finaler Logloss:', losses[-1])
print('Gefundene Theta-Koeffizienten:', theta_gd)


Iter 0: loss=0.580249
Iter 1000: loss=0.019464
Iter 2000: loss=0.015212
Iter 3000: loss=0.013577
Iter 4000: loss=0.012677
Iter 5000: loss=0.012096
Iter 6000: loss=0.011686
Iter 7000: loss=0.011379
Iter 8000: loss=0.011140
Iter 9000: loss=0.010949
Finaler Logloss: 0.010791613156530369
Gefundene Theta-Koeffizienten: [-9.48033964  5.58564503  4.41951298]


In [ ]:
# Plot: Entscheidungsgrenze im standardisierten Merkmalsraum (Torque_z vs RotSpeed_z)
import matplotlib.pyplot as plt

# Scatter der Datenpunkte (Farbe nach Zielvariable y)
plt.figure(figsize=(8,6))
sc = plt.scatter(df['Torque_z'], df['RotSpeed_z'], c=y, cmap='bwr', alpha=0.6, s=18, edgecolor='k', linewidth=0.2)

# Entscheidungsgrenze: p = 0.5 <=> X @ theta = 0
theta = theta_gd

# Gitter für Kontur
tx = np.linspace(df['Torque_z'].min() - 0.5, df['Torque_z'].max() + 0.5, 200)
rx = np.linspace(df['RotSpeed_z'].min() - 0.5, df['RotSpeed_z'].max() + 0.5, 200)
XX, YY = np.meshgrid(tx, rx)
Z = sigmoid(theta[0] + theta[1] * XX + theta[2] * YY)

# Kontur bei p=0.5 (Entscheidungsgrenze)
cont = plt.contour(XX, YY, Z, levels=[0.5], colors='k', linewidths=2)

plt.xlabel('Torque_z')
plt.ylabel('RotSpeed_z')
plt.title('Entscheidungsgrenze für PWF (logistische Regression)')
plt.colorbar(sc, ticks=[0,1], label='PWF')
plt.grid(alpha=0.3)
plt.show()
